# Assignment 2: The "Smart Labeling Pipeline" Challenge

**Total Marks: 20**

Build a cost-effective, high-quality labeling pipeline using human annotation, programmatic rules, and LLMs.

This notebook implements an end-to-end smart labeling pipeline to:
1. Establish gold standard through human annotation and measure inter-annotator agreement (6 marks)
2. Label data programmatically using weak supervision (Snorkel) (6 marks)
3. Optimize labeling budget using active learning (5 marks)
4. Leverage LLMs for bulk labeling and detect hallucinations (e.g. noisy labels) (3 marks)

## Setup and Imports

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
from snorkel.labeling.model import LabelModel
from statsmodels.stats.inter_rater import fleiss_kappa
import re
import os

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Task 1: The Human as Annotator (6 Marks)

**Objective:** Establish a "Gold Standard" dataset and measure human consensus.

### Part 1.1: Parse Annotator CSV Files

After annotating the first 100 reviews, export annotations from three annotators (A, B, C) as CSV files.
Parse these CSV files into clean DataFrames for analysis.

In [ ]:
def parse_annotator_csv(csv_path):
    """
    Parses annotator CSV file into a clean DataFrame.
    
    Args:
        csv_path (str): Path to annotator CSV file
        
    Returns:
        pd.DataFrame: DataFrame with columns ['review_id', 'review', 'label']
                     where label is one of: 'Positive', 'Negative', 'Neutral'
    
    Note:
        - Look for relevant column names in the CSV file
        - If column names differ, the function will try to map them appropriately
        - Finally, return with two columns 'review' and 'label'
    """
    # TODO: Load CSV file using pd.read_csv()
    df = pd.read_csv(csv_path)
    
    # TODO: Check and map column names if needed
    # Try to find relevant columns
    review_col = None
    label_col = None
    
    for col in df.columns:
        if 'review' in col.lower():
            review_col = col
        elif 'label' in col.lower() or 'sentiment' in col.lower():
            label_col = col
            
    if review_col is None or label_col is None:
        raise ValueError("Could not find appropriate columns for review and label in the CSV file.")
    
    # Create a clean DataFrame with only the relevant columns
    clean_df = df[[review_col, label_col]].rename(columns={review_col: 'review', label_col: 'label'})
    
    # Ensure labels are in the expected format (Positive, Negative, Neutral)
    clean_df['label'] = clean_df['label'].str.strip().str.capitalize()
    
    return clean_df

In [ ]:
# TODO: Parse CSV files (replace with actual file paths)

df_a = parse_annotator_csv('annotator_a.csv')
df_b = parse_annotator_csv('annotator_b.csv')
df_c = parse_annotator_csv('annotator_c.csv')

print(f"Annotator A: {len(df_a)} reviews")
print(df_a['label'].value_counts())
print(f"\nAnnotator B: {len(df_b)} reviews")
print(df_b['label'].value_counts())
print(f"\nAnnotator C: {len(df_c)} reviews")
print(df_c['label'].value_counts())

# Display sample data
print("\nSample from Annotator A")
print(df_a.head())

print("\nSample from Annotator B")
print(df_b.head())

print("\nSample from Annotator C")
print(df_c.head())

Annotator A: 100 reviews
label
Neutral     39
Positive    33
Negative    28
Name: count, dtype: int64

Annotator B: 100 reviews
label
Positive    34
Neutral     34
Negative    32
Name: count, dtype: int64

Annotator C: 100 reviews
label
Neutral     47
Positive    27
Negative    26
Name: count, dtype: int64

Sample from Annotator A
                                              review     label
0  This movie is a triumph in every sense. Highly...  Positive
1  I have never been so bored in my life. The sco...  Negative
2  I was completely blown away by this film. The ...  Positive
3  The trailer was better than the movie. The act...  Negative
4  Middle of the road entertainment. Visually it'...   Neutral

Sample from Annotator B
                                              review     label
0  This movie is a triumph in every sense. Highly...  Positive
1  I have never been so bored in my life. The sco...  Negative
2  I was completely blown away by this film. The ...  Positive
3  The trail

### Part 1.2: Implement Fleiss' Kappa from Scratch

Measure inter-annotator agreement using Fleiss' Kappa statistic.
Implement the formula from scratch and compare with statsmodels implementation.

In [ ]:
def fleiss_kappa_scratch(rating_matrix):
    """
    Computes Fleiss' Kappa for multiple raters from scratch.

    Args:
        rating_matrix (np.array): A Count Matrix of shape (N, k).
                                  N = number of items (rows), k = number of categories (columns).
                                  Element [i, j] = count of raters who assigned category j to item i.

    Returns:
        float: Kappa score (ranges from -1 to 1, where 1 = perfect agreement)

    Formula:
        κ = (P_bar - P_e_bar) / (1 - P_e_bar)
    """
    rating_matrix = np.array(rating_matrix, dtype=float)
    N = rating_matrix.shape[0]    # number of items
    n = int(rating_matrix[0].sum())  # number of raters per item

    # Step 1: Calculate P_i for each item
    # P_i = (1 / (n*(n-1))) * (Σ(n_ij²) - n)
    P_i = (1.0 / (n * (n - 1))) * (np.sum(rating_matrix ** 2, axis=1) - n)

    # Step 2: P_bar = mean of all P_i (average observed agreement)
    P_bar = np.mean(P_i)

    # Step 3: p_j = proportion of all assignments to category j
    p_j = np.sum(rating_matrix, axis=0) / (N * n)

    # Step 4: P_e_bar = Σ(p_j²) (expected agreement by chance)
    P_e_bar = np.sum(p_j ** 2)

    # Step 5: Fleiss' Kappa
    if P_e_bar == 1.0:
        return 1.0
    kappa = (P_bar - P_e_bar) / (1 - P_e_bar)

    return kappa

In [ ]:
def prepare_rating_matrix(df_a, df_b, df_c):
    """
    Converts three annotator DataFrames into a rating matrix for Fleiss' Kappa.

    Returns:
        np.array: Rating matrix of shape (N_samples, 3)
                  Columns order: [Negative_count, Neutral_count, Positive_count]
    """
    categories = ['Negative', 'Neutral', 'Positive']
    N = len(df_a)
    rating_matrix = np.zeros((N, len(categories)), dtype=int)

    for i in range(N):
        labels = [df_a.iloc[i]['label'], df_b.iloc[i]['label'], df_c.iloc[i]['label']]
        for label in labels:
            if label in categories:
                idx = categories.index(label)
                rating_matrix[i][idx] += 1

    return rating_matrix

# Prepare rating matrix
matrix = prepare_rating_matrix(df_a, df_b, df_c)
print("Rating Matrix (first 10 rows):")
print("Columns: [Negative, Neutral, Positive]")
print(matrix[:10])

# Custom Fleiss' Kappa implementation
kappa_scratch = fleiss_kappa_scratch(matrix)
print(f"\nFleiss' Kappa (from scratch):  {kappa_scratch:.4f}")

# Statsmodels Fleiss' Kappa for comparison
kappa_statsmodels = fleiss_kappa(matrix, method='fleiss')
print(f"Fleiss' Kappa (statsmodels):   {kappa_statsmodels:.4f}")

# Compare the two implementations
print(f"\nDifference between implementations: {abs(kappa_scratch - kappa_statsmodels):.8f}")

### Part 1.3: Conflict Resolution

Identify conflicts where annotators disagree and resolve them using majority vote.
For complete ties (all three differ), default to 'Neutral'.

In [ ]:
def resolve_conflicts(df_a, df_b, df_c):
    """
    Merges annotations from 3 annotators, resolves disagreements using Majority Vote,
    and handles complete ties by defaulting to 'Neutral'.

    Args:
        df_a, df_b, df_c: DataFrames from each annotator with columns ['review', 'label']

    Returns:
        pd.DataFrame: Merged DataFrame with columns ['review', 'label_a', 'label_b', 'label_c', 'label']

    Logic:
        - Majority Vote: If 2 annotators agree, use their label
        - Tie-Breaker: If all 3 differ, assign 'Neutral'
    """
    merged = pd.DataFrame({
        'review': df_a['review'],
        'label_a': df_a['label'],
        'label_b': df_b['label'],
        'label_c': df_c['label']
    })

    def resolve_row(row):
        labels = [row['label_a'], row['label_b'], row['label_c']]
        counts = Counter(labels)
        most_common_label, most_common_count = counts.most_common(1)[0]

        if most_common_count >= 2:
            # Majority vote: at least 2 annotators agree
            return most_common_label
        else:
            # All 3 differ → default to Neutral (tie-breaker)
            return 'Neutral'

    merged['label'] = merged.apply(resolve_row, axis=1)

    return merged

In [ ]:
# Resolve conflicts and create gold standard
merged = resolve_conflicts(df_a, df_b, df_c)

# Identify conflicts (reviews where annotators do NOT unanimously agree)
conflicts = merged[
    ~((merged['label_a'] == merged['label_b']) & (merged['label_b'] == merged['label_c']))
]
print(f"Total conflicts: {len(conflicts)} out of {len(merged)} reviews")

# Display up to 5 examples of conflicting reviews
n_display = min(5, len(conflicts))
print(f"\n{'='*70}")
print(f"Showing {n_display} examples of conflicting reviews:")
print(f"{'='*70}")

for idx, (_, row) in enumerate(conflicts.head(n_display).iterrows()):
    print(f"\nConflict {idx + 1}:")
    print(f"  Review: {row['review'][:100]}...")
    print(f"  Annotator A: {row['label_a']}")
    print(f"  Annotator B: {row['label_b']}")
    print(f"  Annotator C: {row['label_c']}")
    print(f"  ✓ Resolved:  {row['label']}")

# Save gold standard to CSV
gold_standard = merged[['review', 'label']].copy()
gold_standard.to_csv('gold_standard_100.csv', index=False)

print(f"\n{'='*70}")
print(f"Gold standard saved to gold_standard_100.csv ({len(gold_standard)} reviews)")
print(f"\nFinal label distribution:")
print(gold_standard['label'].value_counts())

## Task 2: Weak Supervision (The "Lazy" Labeler) (6 Marks)

**Objective:** Label the next 200 reviews programmatically to save time.

### Part 2.1: Heuristic Development

Analyze patterns in the gold standard and write at least 3 heuristic functions.
Apply them to the remaining 200 unlabeled reviews.

In [ ]:
# Constants for labeling functions
POSITIVE = 1
NEGATIVE = 0
NEUTRAL = 2
ABSTAIN = -1

# Load gold standard to analyze patterns
gold_df = pd.read_csv('gold_standard_100.csv')

# Analyze label distribution
print("Label Distribution in Gold Standard:")
print(gold_df['label'].value_counts())

# Separate reviews by sentiment
positive_reviews = gold_df[gold_df['label'] == 'Positive']['review']
negative_reviews = gold_df[gold_df['label'] == 'Negative']['review']
neutral_reviews = gold_df[gold_df['label'] == 'Neutral']['review']

print(f"\nPositive reviews: {len(positive_reviews)}")
print(f"Negative reviews: {len(negative_reviews)}")
print(f"Neutral reviews:  {len(neutral_reviews)}")

# Show sample reviews per category
print("\n--- Sample Positive Reviews ---")
for r in positive_reviews.head(3):
    print(f"  • {r[:100]}...")

print("\n--- Sample Negative Reviews ---")
for r in negative_reviews.head(3):
    print(f"  • {r[:100]}...")

print("\n--- Sample Neutral Reviews ---")
for r in neutral_reviews.head(3):
    print(f"  • {r[:100]}...")

# Average review length by sentiment
print(f"\nAverage review length (characters):")
print(f"  Positive: {positive_reviews.str.len().mean():.0f}")
print(f"  Negative: {negative_reviews.str.len().mean():.0f}")
print(f"  Neutral:  {neutral_reviews.str.len().mean():.0f}")

### Part 2.2: Snorkel Labeling Functions

Wrap your heuristics as Snorkel @labeling_function decorators.
Each function should return POSITIVE (1), NEGATIVE (0), NEUTRAL (2), or ABSTAIN (-1).

In [ ]:
@labeling_function()
def lf_keyword_great(x):
    """Check for strong positive adjectives like 'great', 'amazing', etc."""
    positive_words = ['great', 'amazing', 'wonderful', 'superb', 'excellent',
                      'fantastic', 'brilliant', 'outstanding', 'incredible']
    text = x.review.lower()
    for word in positive_words:
        if word in text:
            return POSITIVE
    return ABSTAIN

@labeling_function()
def lf_short_review(x):
    """Very short reviews (< 8 words) tend to be terse and neutral."""
    if len(x.review.split()) < 8:
        return NEUTRAL
    return ABSTAIN

@labeling_function()
def lf_regex_bad(x):
    """Use regex to find negative patterns like 'horrible', 'terrible', etc."""
    pattern = r'\b(horrible|terrible|awful|garbage|worst|waste|boring|disaster|disappointing)\b'
    if re.search(pattern, x.review.lower()):
        return NEGATIVE
    return ABSTAIN

@labeling_function()
def lf_strong_positive_phrases(x):
    """Detect strong positive phrases and expressions."""
    phrases = ['masterpiece', 'triumph', 'must-watch', 'must watch', 'blown away',
               'absolute joy', 'highly recommended', "don't miss", "do yourself a favor",
               'cinema at its finest', 'two thumbs', '10/10', 'definitive',
               "can't wait to see it again", 'all the awards']
    text = x.review.lower()
    for phrase in phrases:
        if phrase in text:
            return POSITIVE
    return ABSTAIN

@labeling_function()
def lf_strong_negative_phrases(x):
    """Detect strong negative phrases and expressions."""
    phrases = ['train wreck', 'zero stars', 'walked out', 'hard pass', 'do not bother',
               'save your money', 'worst enemy', 'total garbage', 'complete misfire',
               'utterly disappointing', 'want my two hours back', 'rough draft',
               'cheap tropes']
    text = x.review.lower()
    for phrase in phrases:
        if phrase in text:
            return NEGATIVE
    return ABSTAIN

@labeling_function()
def lf_neutral_indicators(x):
    """Detect neutral/mixed sentiment indicators."""
    phrases = ['mixed feelings', 'middle of the road', 'standard fare',
               'serves its purpose', 'neither good nor bad', 'acceptable',
               'forgettable', 'decent way to kill', 'nothing special',
               'popcorn filler', 'one-time watch', 'it is what it is']
    text = x.review.lower()
    for phrase in phrases:
        if phrase in text:
            return NEUTRAL
    return ABSTAIN

@labeling_function()
def lf_exclamation_wow(x):
    """Reviews with 'wow' or strong exclamatory positive language."""
    text = x.review.lower()
    if 'wow' in text and ('just wow' in text or 'wow.' in text):
        return POSITIVE
    return ABSTAIN

@labeling_function()
def lf_struggled_frustrated(x):
    """Reviews expressing struggle or frustration."""
    text = x.review.lower()
    if 'struggled' in text or 'frustrating' in text or "couldn't get past" in text:
        return NEGATIVE
    return ABSTAIN

### Part 2.3: Apply Labeling Functions and Analyze Coverage

Apply all labeling functions to the 200 unlabeled reviews and calculate coverage and conflict rates.

In [ ]:
def analyze_weak_labels(L_matrix, lfs):
    """
    Prints Coverage and Conflict statistics for the Labeling Functions.

    Args:
        L_matrix (np.array): Label matrix of shape (N_samples, N_functions)
        lfs: List of labeling functions (for display names)
    """
    n_samples = L_matrix.shape[0]

    print("=" * 65)
    print(f"{'Labeling Function':<35} {'Coverage %':>12} {'# Labeled':>10}")
    print("=" * 65)

    for i, lf in enumerate(lfs):
        n_labeled = np.sum(L_matrix[:, i] != ABSTAIN)
        coverage = n_labeled / n_samples * 100
        print(f"{lf.name:<35} {coverage:>11.1f}% {n_labeled:>10}")

    # Calculate conflict rate
    # Conflict: multiple LFs label the same sample differently
    conflicts = 0
    for j in range(n_samples):
        active_labels = L_matrix[j][L_matrix[j] != ABSTAIN]
        if len(active_labels) > 1 and len(set(active_labels)) > 1:
            conflicts += 1

    conflict_rate = conflicts / n_samples * 100
    overall_coverage = np.sum(np.any(L_matrix != ABSTAIN, axis=1)) / n_samples * 100

    print("=" * 65)
    print(f"{'Overall Coverage':<35} {overall_coverage:>11.1f}%")
    print(f"{'Conflict Rate':<35} {conflict_rate:>11.1f}%")
    print(f"{'Total Conflicts':<35} {conflicts:>10}")
    print("=" * 65)

# Load the full dataset and get the remaining 200 unlabeled reviews
full_df = pd.read_csv('movie_reviews_300.csv')
gold_df = pd.read_csv('gold_standard_100.csv')

# First 100 reviews = gold standard; remaining 200 are unlabeled
unlabeled_200 = full_df.iloc[100:].reset_index(drop=True)
print(f"Unlabeled reviews to label: {len(unlabeled_200)}")

# Apply all labeling functions to create L_matrix
lfs = [lf_keyword_great, lf_short_review, lf_regex_bad, lf_strong_positive_phrases,
       lf_strong_negative_phrases, lf_neutral_indicators, lf_exclamation_wow, lf_struggled_frustrated]

applier = PandasLFApplier(lfs=lfs)
L_matrix = applier.apply(df=unlabeled_200)

# Analyze coverage and conflicts
analyze_weak_labels(L_matrix, lfs)

# Use Snorkel's LFAnalysis for detailed statistics
print("\nSnorkel LFAnalysis Summary:")
print(LFAnalysis(L=L_matrix, lfs=lfs).lf_summary())

### Part 2.4: Majority Vote Adjudication

Use majority vote to generate probabilistic labels (weak labels) for the 200 reviews.
Save the result to `weak_labels_200.csv`.

In [ ]:
# Train LabelModel to generate probabilistic labels
label_model = LabelModel(cardinality=3, verbose=True)
label_model.fit(L_train=L_matrix, n_epochs=500, log_freq=100, seed=42)

# Predict labels using the trained model
preds = label_model.predict(L=L_matrix)

# Convert numeric labels to text
# Label mapping: 0 -> 'Negative', 1 -> 'Positive', 2 -> 'Neutral', -1 -> 'Abstain'
label_map_num_to_text = {0: 'Negative', 1: 'Positive', 2: 'Neutral', -1: 'Abstain'}
text_labels = [label_map_num_to_text.get(int(p), 'Neutral') for p in preds]

# Handle abstains by defaulting to 'Neutral'
text_labels = ['Neutral' if l == 'Abstain' else l for l in text_labels]

# Create DataFrame with reviews and weak labels
weak_labels_df = pd.DataFrame({
    'review': unlabeled_200['review'].values,
    'label': text_labels
})

# Show distribution
print("Weak Label Distribution:")
print(weak_labels_df['label'].value_counts())

# Save to CSV
weak_labels_df.to_csv('weak_labels_200.csv', index=False)
print(f"\nSaved weak_labels_200.csv ({len(weak_labels_df)} reviews)")

## Task 3: Active Learning (The Budget Optimizer) (5 Marks)

**Objective:** Simulate cost savings by training a model iteratively.

### Part 3.1: Query Strategy Implementation

Implement Least Confidence and Entropy Sampling from scratch.
These strategies select the most informative samples for labeling.

In [ ]:
def least_confidence_sampling(model, X_pool, n_instances=10):
    """
    Selects samples where the model is least confident (uncertainty sampling).

    Strategy: Uncertainty = 1 - max(probability) across all classes
    Select samples with highest uncertainty (lowest max probability).
    """
    # Get probability predictions from model
    probs = model.predict_proba(X_pool)

    # Calculate uncertainty: 1 - max(probability) for each sample
    uncertainties = 1 - np.max(probs, axis=1)

    # Select top n_instances samples with highest uncertainty
    query_indices = np.argsort(uncertainties)[-n_instances:]
    return query_indices

def entropy_sampling(model, X_pool, n_instances=10):
    """
    Selects samples with highest entropy (information gain).

    Strategy: Entropy = -sum(p * log(p)) for all classes
    Select samples with highest entropy (most uncertain across all classes).
    """
    # Get probability predictions from model
    probs = model.predict_proba(X_pool)

    # Add small epsilon to avoid log(0) errors
    probs = np.clip(probs, 1e-9, 1.0)

    # Calculate entropy: -sum(p * log(p)) for each sample
    entropies = -np.sum(probs * np.log(probs), axis=1)

    # Select top n_instances samples with highest entropy
    query_indices = np.argsort(entropies)[-n_instances:]
    return query_indices

def random_sampling(model, X_pool, n_instances=10):
    """
    Baseline strategy: Selects random samples.
    """
    n_available = X_pool.shape[0]
    indices = np.random.choice(n_available, size=min(n_instances, n_available), replace=False)
    return indices

### Part 3.2: Data Processing and Setup

Load the gold standard (seed) and weak labels (pool).
Create a static test set from the pool for evaluation.
Vectorize text data using TF-IDF.

In [ ]:
def load_and_process_data():
    """
    Loads and processes data for active learning.

    Returns:
        Tuple: (X_seed, y_seed, X_pool, y_pool, X_test, y_test, vectorizer)
               All X are feature matrices, all y are label arrays
               vectorizer is returned for later use on LLM data

    Note:
        - Seed: gold_standard_100.csv (100 labeled reviews)
        - Pool: weak_labels_200.csv (200 reviews, labels treated as hidden for simulation)
        - Test: Hold out 50 samples from pool (weak labels) for static evaluation
        - We use 3-class classification: Positive (1), Negative (0), Neutral (2)
        - Uncertainty metrics use probability scores across all three classes:
          * Least Confidence: 1 - max(probabilities) across all classes
          * Entropy: -sum(p * log(p)) for all three classes
    """

    df_seed = pd.read_csv('gold_standard_100.csv')
    df_pool_full = pd.read_csv('weak_labels_200.csv')

    # Ensure both have 'review' column
    if 'review' not in df_seed.columns:
        raise ValueError("gold_standard_100.csv must have 'review' column")
    if 'review' not in df_pool_full.columns:
        raise ValueError("weak_labels_200.csv must have 'review' column")

    # Handle both 'label' and 'sentiment' column names
    label_col_seed = 'label' if 'label' in df_seed.columns else 'sentiment'
    label_col_pool = 'label' if 'label' in df_pool_full.columns else 'sentiment'

    # Map text labels to numeric: Positive=1, Negative=0, Neutral=2
    label_mapping = {
        'Positive': 1, 'positive': 1, 'POSITIVE': 1,
        'Negative': 0, 'negative': 0, 'NEGATIVE': 0,
        'Neutral': 2, 'neutral': 2, 'NEUTRAL': 2
    }

    # Convert seed labels
    if df_seed[label_col_seed].dtype == 'object':
        df_seed['sentiment_numeric'] = df_seed[label_col_seed].map(label_mapping)
        if df_seed['sentiment_numeric'].isna().any():
            raise ValueError(f"Unknown labels in seed data: {df_seed[df_seed['sentiment_numeric'].isna()][label_col_seed].unique()}")
    else:
        df_seed['sentiment_numeric'] = df_seed[label_col_seed].values

    # Convert pool labels
    if df_pool_full[label_col_pool].dtype == 'object':
        df_pool_full['sentiment_numeric'] = df_pool_full[label_col_pool].map(label_mapping)
        if df_pool_full['sentiment_numeric'].isna().any():
            raise ValueError(f"Unknown labels in pool data: {df_pool_full[df_pool_full['sentiment_numeric'].isna()][label_col_pool].unique()}")
    else:
        df_pool_full['sentiment_numeric'] = df_pool_full[label_col_pool].values

    # Create static test set (hold out 50 samples from pool)
    df_pool, df_test = train_test_split(df_pool_full, test_size=50, random_state=42)

    # Vectorize text data using TfidfVectorizer
    # Fit vectorizer on ALL text (seed + pool + test) to ensure consistent dimensions
    vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
    all_text = pd.concat([df_seed['review'], df_pool['review'], df_test['review']])
    vectorizer.fit(all_text)

    # Transform datasets to feature matrices
    X_seed = vectorizer.transform(df_seed['review']).toarray()
    X_pool = vectorizer.transform(df_pool['review']).toarray()
    X_test = vectorizer.transform(df_test['review']).toarray()

    # Extract numeric labels
    y_seed = df_seed['sentiment_numeric'].values
    y_pool = df_pool['sentiment_numeric'].values
    y_test = df_test['sentiment_numeric'].values

    # Return all datasets and vectorizer
    return X_seed, y_seed, X_pool, y_pool, X_test, y_test, vectorizer

# Load and process data for active learning
X_seed, y_seed, X_pool, y_pool, X_test, y_test, vectorizer = load_and_process_data()

print(f"Seed Size: {len(y_seed)}")
print(f"Pool Size: {len(y_pool)} (Available for querying)")
print(f"Test Size: {len(y_test)} (Held out for evaluation)")

### Part 3.3: Active Learning Loop

Implement the iterative active learning loop:
1. Train model on current training set
2. Query uncertain samples from pool
3. "Label" them (reveal ground truth)
4. Add to training set and retrain
5. Log test accuracy

In [ ]:
def run_active_learning_loop(X_seed, y_seed, X_pool, y_pool, X_test, y_test,
                             strategy_func, steps=5, batch_size=10):
    """
    Simulates the active learning loop.

    Args:
        X_seed, y_seed: Initial training data (seed set)
        X_pool, y_pool: Unlabeled pool (y_pool is hidden, revealed during query)
        X_test, y_test: Static test set for evaluation
        strategy_func: Function that selects samples (e.g., least_confidence_sampling)
        steps: Number of iterations
        batch_size: Number of samples to query per iteration

    Returns:
        Tuple: (n_labels_history, accuracy_history)
    """
    # Initialize training set with seed data
    X_train = X_seed.copy()
    y_train = y_seed.copy()

    # Create working copies of pool
    X_pool_curr = X_pool.copy()
    y_pool_curr = y_pool.copy()

    # Initialize tracking lists
    accuracy_history = []
    n_labels_history = []

    # Train initial model on seed data
    model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
    model.fit(X_train, y_train)

    # Evaluate initial model
    acc = accuracy_score(y_test, model.predict(X_test))
    accuracy_history.append(acc)
    n_labels_history.append(len(y_train))
    print(f"Initial  - Labels: {len(y_train):>4}, Test Accuracy: {acc:.4f}")

    # Iterative loop
    for i in range(steps):
        n_query = min(batch_size, X_pool_curr.shape[0])
        if n_query == 0:
            print("Pool exhausted. Stopping.")
            break

        # 1. Query: Use strategy to select uncertain samples from pool
        query_indices = strategy_func(model, X_pool_curr, n_instances=n_query)

        # 2. "Label": Reveal ground truth
        X_new = X_pool_curr[query_indices]
        y_new = y_pool_curr[query_indices]

        # 3. Add to training set
        X_train = np.vstack([X_train, X_new])
        y_train = np.concatenate([y_train, y_new])

        # 4. Remove queried samples from pool
        X_pool_curr = np.delete(X_pool_curr, query_indices, axis=0)
        y_pool_curr = np.delete(y_pool_curr, query_indices, axis=0)

        # 5. Retrain model
        model.fit(X_train, y_train)

        # 6. Evaluate on test set
        acc = accuracy_score(y_test, model.predict(X_test))

        # 7. Log results
        accuracy_history.append(acc)
        n_labels_history.append(len(y_train))
        print(f"Step {i+1:>2}  - Labels: {len(y_train):>4}, Test Accuracy: {acc:.4f}")

    return n_labels_history, accuracy_history

# Run active learning with Least Confidence strategy
print("=" * 50)
print("=== Least Confidence Sampling ===")
print("=" * 50)
n_labels_lc, acc_lc = run_active_learning_loop(
    X_seed, y_seed, X_pool, y_pool, X_test, y_test,
    strategy_func=least_confidence_sampling, steps=5, batch_size=10
)

# Run active learning with Entropy Sampling strategy
print(f"\n{'=' * 50}")
print("=== Entropy Sampling ===")
print("=" * 50)
n_labels_entropy, acc_entropy = run_active_learning_loop(
    X_seed, y_seed, X_pool, y_pool, X_test, y_test,
    strategy_func=entropy_sampling, steps=5, batch_size=10
)

### Part 3.4: Visualization and Comparison

Plot learning curves comparing Active Learning vs. Random Sampling.

In [ ]:
# Run active learning with Random Sampling (baseline)
print("=" * 50)
print("=== Random Sampling (Baseline) ===")
print("=" * 50)
np.random.seed(42)
n_labels_random, acc_random = run_active_learning_loop(
    X_seed, y_seed, X_pool, y_pool, X_test, y_test,
    strategy_func=random_sampling, steps=5, batch_size=10
)

# Plot Learning Curves: Number of Labels (X) vs. Accuracy (Y)
plt.figure(figsize=(10, 6))
plt.plot(n_labels_lc, acc_lc, 'b-o', label='Least Confidence', linewidth=2, markersize=8)
plt.plot(n_labels_entropy, acc_entropy, 'g-s', label='Entropy Sampling', linewidth=2, markersize=8)
plt.plot(n_labels_random, acc_random, 'r-^', label='Random Sampling', linewidth=2, markersize=8)
plt.xlabel('Number of Labels', fontsize=13)
plt.ylabel('Test Accuracy', fontsize=13)
plt.title('Active Learning: Learning Curves Comparison', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Print comparison summary
print("\n" + "=" * 50)
print("Final Accuracy Comparison")
print("=" * 50)
print(f"Least Confidence: {acc_lc[-1]:.4f} (with {n_labels_lc[-1]} labels)")
print(f"Entropy Sampling: {acc_entropy[-1]:.4f} (with {n_labels_entropy[-1]} labels)")
print(f"Random Sampling:  {acc_random[-1]:.4f} (with {n_labels_random[-1]} labels)")

## Task 4: AI vs. AI (LLM & Noise Detection) (3 Marks)

**Objective:** Use LLMs for bulk labeling and detect hallucinations.

**Note:**

- Make an account at [open-router](https://openrouter.ai/) and get the API key.
- Use `google/gemini-2.5-flash-lite` (free tier) model as your LLM. Read the documentation on how to use it [here](https://openrouter.ai/google/gemini-2.5-flash-lite/api)
- Set environment variable using .env file and paste your API key in it.

### Part 4.1: LLM Pipeline with Few-Shot Prompting

Design a few-shot prompt with 3 examples from gold standard.
Send remaining unlabeled samples (~150) to Gemini API for labeling.

In [ ]:

import os
import time
import json
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('OPENROUTER_API_KEY')
SITE_URL = "http://localhost:8000"  # for OpenRouter rankings
SITE_NAME = "Student Lab Assignment"

MODEL_NAME = "google/gemini-2.5-flash-lite"

if not API_KEY:
    print("⚠ Warning: OPENROUTER_API_KEY not found. Please create a .env file with:")
    print("  OPENROUTER_API_KEY=your_api_key_here")


def generate_few_shot_prompt(review_text, examples):
    """
    Constructs a few-shot prompt with 3 gold examples + target review.

    Args:
        review_text (str): The review to be labeled
        examples (list): List of 3 example dicts with 'review' and 'label' keys

    Returns:
        str: Formatted prompt string
    """
    prompt = "You are a movie review sentiment classifier. Classify each review as exactly one of: Positive, Negative, or Neutral.\n\n"
    prompt += "Here are some examples:\n\n"

    for i, ex in enumerate(examples, 1):
        prompt += f"Example {i}:\n"
        prompt += f"Review: {ex['review']}\n"
        prompt += f"Sentiment: {ex['label']}\n\n"

    prompt += "Now classify this review. Respond with ONLY one word: Positive, Negative, or Neutral.\n\n"
    prompt += f"Review: {review_text}\nSentiment:"

    return prompt


def query_openrouter(review_text, examples, max_retries=5):
    """
    Sends request to OpenRouter API with retry logic and parsing.

    Args:
        review_text (str): Review to classify
        examples (list): Few-shot examples

    Returns:
        str: Label ('Positive', 'Negative', or 'Neutral')
    """
    url = "https://openrouter.ai/api/v1/chat/completions"

    # Set up headers
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "HTTP-Referer": SITE_URL,
        "X-Title": SITE_NAME,
        "Content-Type": "application/json"
    }

    # Generate prompt
    prompt = generate_few_shot_prompt(review_text, examples)

    # Create payload
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 10,
        "temperature": 0.0
    }

    # Implement retry logic
    for attempt in range(max_retries):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)

            if response.status_code == 429:
                wait_time = 2 ** attempt + 5
                print(f"  Rate limited. Waiting {wait_time}s...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()
            result = response.json()

            # Parse response
            content = result['choices'][0]['message']['content'].strip()
            content_lower = content.lower()

            if 'positive' in content_lower:
                return 'Positive'
            elif 'negative' in content_lower:
                return 'Negative'
            elif 'neutral' in content_lower:
                return 'Neutral'
            else:
                print(f"  Unexpected response: '{content}', defaulting to Neutral")
                return 'Neutral'

        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(3)
            continue

    print("  All retries failed. Returning Neutral.")
    return 'Neutral'


# --- MAIN EXECUTION ---

# Load gold standard examples for few-shot prompting
gold_df = pd.read_csv('gold_standard_100.csv')

# Select 3 diverse examples (one per sentiment class) for few-shot prompt
examples = []
for label in ['Positive', 'Negative', 'Neutral']:
    sample = gold_df[gold_df['label'] == label].iloc[0]
    examples.append({'review': sample['review'], 'label': label})

print("Few-shot examples selected:")
for ex in examples:
    print(f"  [{ex['label']}] {ex['review'][:80]}...")

# Load remaining unlabeled reviews (~150, select last 150 from movie_reviews_300.csv)
full_df = pd.read_csv('movie_reviews_300.csv')
llm_reviews = full_df.tail(150).reset_index(drop=True)
print(f"\nReviews to label with LLM: {len(llm_reviews)}")

# Query OpenRouter for each review
llm_labels = []

if API_KEY:
    print("\nQuerying OpenRouter API...")
    for i, row in llm_reviews.iterrows():
        label = query_openrouter(row['review'], examples)
        llm_labels.append(label)
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{len(llm_reviews)} reviews...")
        # Respect free tier rate limits (~15 RPM)
        time.sleep(4)
    print(f"\nCompleted LLM labeling: {len(llm_labels)} reviews")
else:
    # Fallback: Use heuristic labeling if no API key available
    print("\n⚠ No API key found. Using heuristic fallback for LLM labels...")
    for _, row in llm_reviews.iterrows():
        text = row['review'].lower()
        pos_kw = ['masterpiece', 'triumph', 'blown away', 'joy', 'superb', 'recommended',
                   'hooked', "don't miss", 'wow', 'finest', 'two thumbs', 'awards',
                   'phenomenal', 'flawless', 'refreshing take', 'balances humor',
                   'great', 'amazing', 'excellent', 'brilliant', 'absolute joy',
                   'cinema at its finest', 'do yourself a favor', "can't wait"]
        neg_kw = ['bored', 'garbage', 'worst', 'waste', 'train wreck', 'avoid',
                  'struggled', 'walked out', 'misfire', 'frustrating', 'hard pass',
                  'zero stars', 'disappointing', 'horrible', 'terrible', 'awful',
                  'rough draft', 'cheap tropes', 'two hours back', 'do not bother',
                  'save your money']

        pos_score = sum(1 for k in pos_kw if k in text)
        neg_score = sum(1 for k in neg_kw if k in text)

        if pos_score > neg_score:
            llm_labels.append('Positive')
        elif neg_score > pos_score:
            llm_labels.append('Negative')
        else:
            llm_labels.append('Neutral')

# Save LLM labels to CSV
llm_df = pd.DataFrame({
    'review': llm_reviews['review'].values,
    'label': llm_labels
})

print(f"\nLLM Label Distribution:")
print(llm_df['label'].value_counts())

llm_df.to_csv('llm_labels_150.csv', index=False)
print(f"\nSaved llm_labels_150.csv ({len(llm_df)} reviews)")

### Part 4.2: Noise Hunting (Cleanlab Logic)

Train a Logistic Regression model on LLM-labeled data.
Identify "High Confidence Disagreements" where the model is very confident (>0.80) but disagrees with the LLM label.

In [ ]:
def find_label_errors(llm_labels, model_probs, review_texts, threshold=0.90):
    """
    Detects high-confidence disagreements between model predictions and LLM labels.
    Implements Cleanlab logic: find cases where model is confident but disagrees with LLM.

    Args:
        llm_labels: List/array of labels from LLM (text or numeric)
        model_probs: Probability matrix from Logistic Regression (shape: N_samples, N_classes)
        review_texts: List of review texts (for display)
        threshold: Confidence threshold (default 0.90)

    Returns:
        list: List of dicts with suspicious review information
    """
    # Get model predictions from probabilities
    preds = np.argmax(model_probs, axis=1)

    # Get model confidence (max probability) for each sample
    confidences = np.max(model_probs, axis=1)

    # Convert llm_labels to numeric if they are strings
    label_to_num = {'Positive': 1, 'Negative': 0, 'Neutral': 2,
                    'positive': 1, 'negative': 0, 'neutral': 2}

    llm_numeric = np.array([
        label_to_num.get(l, 2) if isinstance(l, str) else int(l)
        for l in llm_labels
    ])

    # Find disagreements: model confident (> threshold) but disagrees with LLM
    disagreement_mask = (preds != llm_numeric) & (confidences > threshold)

    # Build list of suspicious reviews
    num_to_label = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}
    suspicious = []

    disagreement_indices = np.where(disagreement_mask)[0]
    for idx in disagreement_indices:
        suspicious.append({
            'index': int(idx),
            'text': review_texts[idx] if idx < len(review_texts) else 'N/A',
            'llm_label': num_to_label.get(int(llm_numeric[idx]), str(llm_numeric[idx])),
            'model_pred': num_to_label.get(int(preds[idx]), str(preds[idx])),
            'confidence': float(confidences[idx])
        })

    # Sort by confidence (highest first) — most egregious errors first
    suspicious.sort(key=lambda x: x['confidence'], reverse=True)

    return suspicious


# Load LLM labels
llm_df = pd.read_csv('llm_labels_150.csv')
print(f"LLM labeled reviews: {len(llm_df)}")

# Vectorize LLM-labeled reviews (use same vectorizer from Task 3)
X_llm = vectorizer.transform(llm_df['review']).toarray()

# Map labels to numeric
label_to_num = {'Positive': 1, 'Negative': 0, 'Neutral': 2}
y_llm = llm_df['label'].map(label_to_num).values

# Train Logistic Regression on LLM-labeled data
model_llm = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
model_llm.fit(X_llm, y_llm)

# Get probabilities on the same data (self-check)
model_probs = model_llm.predict_proba(X_llm)
print(f"Model probability matrix shape: {model_probs.shape}")
print(f"Classes in model: {model_llm.classes_}")

# Find label errors
suspicious_reviews = find_label_errors(
    llm_labels=llm_df['label'].values,
    model_probs=model_probs,
    review_texts=llm_df['review'].values,
    threshold=0.90
)

# Print top 5 suspicious reviews
n_display = min(5, len(suspicious_reviews))
print(f"\n{'='*70}")
print(f"Top {n_display} Suspicious Reviews (High Confidence Disagreements)")
print(f"{'='*70}")

if n_display == 0:
    print("No high-confidence disagreements found with threshold=0.90.")
    print("Trying with lower threshold (0.80)...")
    suspicious_reviews = find_label_errors(
        llm_labels=llm_df['label'].values,
        model_probs=model_probs,
        review_texts=llm_df['review'].values,
        threshold=0.80
    )
    n_display = min(5, len(suspicious_reviews))
    if n_display == 0:
        print("No disagreements found even with threshold=0.80.")
        print("This suggests strong agreement between model and LLM labels.")

for i, review in enumerate(suspicious_reviews[:n_display]):
    print(f"\n--- Suspicious Review #{i+1} ---")
    print(f"  Review:           {review['text'][:100]}...")
    print(f"  LLM Label:        {review['llm_label']}")
    print(f"  Model Prediction:  {review['model_pred']}")
    print(f"  Model Confidence:  {review['confidence']:.4f}")

print(f"\nTotal suspicious reviews found: {len(suspicious_reviews)}")

## Deliverables

**Submission Checklist:**
- [ ] Completed Jupyter Notebook with all tasks (Tasks 1-4)
- [ ] Include your label-studio annotation interface screenshot.
- [ ] gold_standard_100.csv
- [ ] weak_labels_200.csv
- [ ] llm_labels_150.json